In [71]:
import ee
import geemap
import pandas as pd
import numpy as np
# import numpy as np
# import os
# import seaborn as sns

ee.Authenticate()
ee.Initialize(project='ee-ivanburgov666')

In [72]:
greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)

landmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0).And(ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ice_mask').eq(0))

icemask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ice_mask').eq(1)

greenland = ee.Geometry.Polygon(
    [[[-36.29516924635421, 83.70737243835941],
    [-51.85180987135421, 82.75597137647488],
    [-61.43188799635421, 81.99879137488564],
    [-74.08813799635422, 78.10103528196419],
    [-70.13305987135422, 75.65372336709613],
    [-61.08032549635421, 75.71891096312955],
    [-52.20337237135421, 60.9795530382023],
    [-43.41430987135421, 58.59235996703347],
    [-38.49243487135421, 64.70478286561182],
    [-19.771731746354217, 69.72271161037442],
    [-15.728762996354217, 76.0828635948066],
    [-15.904544246354217, 79.45091003031243],
    [-10.015872371354217, 81.62328742628017],
    [-26.627200496354217, 83.43179828852398],
    [-31.636966121354217, 83.7553561747887]]])

In [73]:
date_start = '2017-06-01'
date_end = '2019-12-01'
single_date = '2020-12-19'

def maskJaxa(image):
    # '''Function to filter JAXA GCOM-C LST data based on quality flag.'''
    # qa = image.select('LST_QA_flag')
    # #0: water (land fraction = 0%)
    # #1: mostly water (0% < land fraction < 50%)
    # #2: mostly coastal (50% < land fraction < 100%) - included
    # #3: land (land fraction = 100%) - included
    # mask = qa.gt(1)
    return image.updateMask(icemask) 

# JAXA
def lst_jaxa_a(image):
    'JAXA GCOM-C band selection and conversion'
    lst_ave = image.select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_A')
    qa_flag = image.select('LST_QA_flag')
    return image.addBands(lst_ave).addBands(qa_flag)

def lst_jaxa_d(image):
    'JAXA GCOM-C band selection and conversion'
    lst_ave = image.select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_D')
    qa_flag = image.select('LST_QA_flag')
    return image.addBands(lst_ave).addBands(qa_flag)

JAXA_A = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'A')) # Filter for ascending (AM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) 
    .filterDate(date_start, date_end)
    # .filterDate(single_date)
    .filterBounds(greenland)
    .map(maskJaxa)
    .map(lst_jaxa_a)
)

JAXA_D = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'D')) # Filter for descending (PM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) 
    .filterDate(date_start, date_end)
    # .filterDate(single_date)
    .filterBounds(greenland)
    .map(maskJaxa)
    .map(lst_jaxa_d)
    )

In [55]:
JAXA_A = JAXA_A.mean()
JAXA_D = JAXA_D.mean()

In [77]:
img = ee.Image('JAXA/GCOM-C/L3/LAND/LST/V3/20180217A')
img = img.select('LST_AVE').multiply(0.02).subtract(273.15).rename('LST_AVE')

In [81]:
JAXA_A
JAXA_D

In [80]:
Map = geemap.Map()
Map.centerObject(greenland, 3)
# Map.addLayer(JAXA_A.select('JAXA_LST_A'), {'min': -30, 'max': 0, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}, 'JAXA LST A')
# Map.addLayer(JAXA_D.select('JAXA_LST_D'), {'min': -30, 'max': 0, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}, 'JAXA LST D')
Map.addLayer(img.select('LST_AVE'), {'min': -0, 'max': 300, 'palette': ['blue', 'cyan', 'green', 'yellow', 'red']}, 'JAXA LST Single Date')


Map

Map(center=[72.70302525720832, -41.78688985236265], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:

import ee
import geemap
import pandas as pd
import numpy as np
# import numpy as np
# import os
# import seaborn as sns

ee.Authenticate()
ee.Initialize(project='ee-ivanburgov666')

# %%
greenlandmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0)

landmask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ocean_mask').eq(0).And(ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ice_mask').eq(0))

icemask = ee.Image('OSU/GIMP/2000_ICE_OCEAN_MASK').select('ice_mask').eq(1)

greenland = ee.Geometry.Polygon(
    [[[-36.29516924635421, 83.70737243835941],
    [-51.85180987135421, 82.75597137647488],
    [-61.43188799635421, 81.99879137488564],
    [-74.08813799635422, 78.10103528196419],
    [-70.13305987135422, 75.65372336709613],
    [-61.08032549635421, 75.71891096312955],
    [-52.20337237135421, 60.9795530382023],
    [-43.41430987135421, 58.59235996703347],
    [-38.49243487135421, 64.70478286561182],
    [-19.771731746354217, 69.72271161037442],
    [-15.728762996354217, 76.0828635948066],
    [-15.904544246354217, 79.45091003031243],
    [-10.015872371354217, 81.62328742628017],
    [-26.627200496354217, 83.43179828852398],
    [-31.636966121354217, 83.7553561747887]]])

# create a vector of month time steps
months = np.arange(1, 13, 1) # last month not inclusive! # 1,13,1
date_start =ee.Date('2000-01-01')
date_end = ee.Date('2025-12-31')

date_end_mod = ee.Date('2025-12-31') # Before orbital drift TERRA 2020-02-27
date_end_myd = ee.Date('2025-12-31') # Before orbital drift AQUA 2021-03-18


lookup_ice = ee.FeatureCollection('projects/ee-ivanburgov666/assets/coefficients_ice_alle')
lookup_land = ee.FeatureCollection('projects/ee-ivanburgov666/assets/coefficients_land_doublettes')


# %%
# Mask data based on quality flags

def bitwiseExtract(input, fromBit, toBit):
    maskSize = ee.Number(1).add(toBit).subtract(fromBit)
    mask = ee.Number(1).leftShift(maskSize).subtract(1)
    return input.rightShift(fromBit).bitwiseAnd(mask)

def maskQualityDaytime(image):
    qa = image.select('QC_Day')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)

def maskQualityNighttime(image):
    qa = image.select('QC_Night')

    # Bits 0-"1": Mandatory QA flags
    # "0": LST produced, good quality, not necessary to examine more detailed QA
    # "1": LST produced, other quality, recommend examination of more detailed QA
    # "2": LST not produced due to cloud effects
    # "3": LST not produced primarily due to reasons other than cloud
    bits01Mask = bitwiseExtract(qa, 0, 1).lte(1); 
    # Bits 2-"3": Data quality flag
    # "0": Good data quality
    # "1": Other quality data
    # "2": TBD
    # "3": TBD
    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 4-"5": Emissivity error flag
    # "0": Average emissivity error <= 0.01
    # "1": 0.01 < Average emissivity error <= 0.02
    # "2": 0.02 < Average emissivity error <= 0.04
    # "3": Average emissivity error > 0.04
    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bit 6-"7": LST error flag
    # "0": Average LST error <= 1K
    # "1": Average LST error <= 2K
    # "2": Average LST error <= 3K
    # "3": Average LST error > 3K
    bit6Mask = bitwiseExtract(qa, 6, 7).lte(1)

    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit6Mask)

    return image.updateMask(mask)


def maskJaxa(image):
    # '''Function to filter JAXA GCOM-C LST data based on quality flag.'''
    # qa = image.select('LST_QA_flag')
    # #0: water (land fraction = 0%)
    # #1: mostly water (0% < land fraction < 50%)
    # #2: mostly coastal (50% < land fraction < 100%) - included
    # #3: land (land fraction = 100%) - included
    # mask = qa.gt(1)
    return image.updateMask(icemask) 

def maskViirs(image):
    '''Function to filter VIIRS LST data based on quality flag.'''
    qa = image.select('QC')
    bits01Mask = bitwiseExtract(qa, 0, 1).eq(0); 
    # Bits 0-1: Mandatory QA flags
    # 0: Pixel produced, good quality, no further QA info necessary
    # 1: Pixel produced but unreliable quality
    # 2: Pixel not produced due to cloud
    # 3: Pixel not produced due to reasons other than cloud

    bits23Mask = bitwiseExtract(qa, 2, 3).eq(0)
    # Bits 2-3: Data quality flag
    # 0: Good data quality of L1B bands 14, 15, 16
    # 1: Missing pixel
    # 2: Fairly calibrated
    # 3: Poorly calibrated, TES processing skipped

    bits45Mask = bitwiseExtract(qa, 4, 5).eq(0)
    # Bits 4-5: Cloud Flag
    # 0: Cloud-free
    # 1: Thin cirrus
    # 2: Pixel within 2 pixels of nearest cloud
    # 3: Cloudy pixels

    # Bits 6-7: Iterations
    # 0: Slow convergence
    # 1: Nominal
    # 2: Nominal
    # 3: Fast

    # Bits 8-9: Atmospheric Opacity
    # 0: ≥3 (Warm, humid air; or cold land)
    # 1: 0.2 - 0.3 (Nominal value)
    # 2: 0.1 - 0.2 (Nominal value)
    # 3: <0.1 (Dry, or high altitude pixel)

    # Bits 10-11: MMD
    # 0: >0.15 (Most silicate rocks)
    # 1: 0.1 - 0.15 (Rocks, sand, some soils)
    # 2: 0.03 - 0.1 (Mostly soils, mixed pixel)
    # 3: <0.03 (Vegetation, snow, water, ice, some soils)

    bit1213Mask = bitwiseExtract(qa, 12, 13).gte(2)
    # Bits 12-13: Emissivity accuracy
    # 0: >0.02 (Poor performance)
    # 1: 0.015 - 0.02 (Marginal performance)
    # 2: 0.01 - 0.015 (Good performance)
    # 3: <0.01 (Excellent performance)

    bit1415Mask = bitwiseExtract(qa, 14, 15).gte(2)
    # Bits 14-15: LST accuracy
    # 0: >2K (Poor performance)
    # 1: 1.5 - 2K (Marginal performance)
    # 2: 1 - 1.5K (Good performance)
    # 3: <1K (Excellent performance)
    
    mask = bits01Mask.And(bits23Mask).And(bits45Mask).And(bit1213Mask).And(bit1415Mask)
    return image.updateMask(mask)



# %%
# Functions for band selection and conversion

# ERA5 (full scale)
def era5_t2m(image):
    'ERA5 2m air temperature conversion'
    t2m = image.select('temperature_2m').subtract(273.15).rename('ERA5_T2m')
    return image.addBands(t2m)

# MODIS
def lst_mod_day(image):
    'Terra Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Day')
    qa_day = image.select('QC_Day').rename('MOD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_mod_night(image):
    'Terra Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MOD_LST_Night')
    qa_night = image.select('QC_Night').rename('MOD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)

def lst_myd_day(image):
    'Aqua Day band selection and conversion'
    lst_day = image.select('LST_Day_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Day')
    qa_day = image.select('QC_Day').rename('MYD_QA_Day')
    return image.addBands(lst_day).addBands(qa_day)

def lst_myd_night(image):
    'Aqua Night band selection and conversion'
    lst_night = image.select('LST_Night_1km').multiply(0.02).subtract(273.15).rename('MYD_LST_Night')
    qa_night = image.select('QC_Night').rename('MYD_QA_Night')
    return image.addBands(lst_night).addBands(qa_night)


# JAXA
def lst_jaxa_a(image):
    'JAXA GCOM-C band selection and conversion'
    lst_ave = image.select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_A')
    qa_flag = image.select('LST_QA_flag')
    return image.addBands(lst_ave).addBands(qa_flag)

def lst_jaxa_d(image):
    'JAXA GCOM-C band selection and conversion'
    lst_ave = image.select('LST_AVE').multiply(0.02).subtract(273.15).rename('JAXA_LST_D')
    qa_flag = image.select('LST_QA_flag')
    return image.addBands(lst_ave).addBands(qa_flag)


# VIIRS 
def lst_viirs_d(image):
    'VIIRS band selection and conversion'
    lst = image.select('LST_1KM').subtract(273.15).rename('VIIRS_LST_D')
    return image.addBands(lst)

def lst_viirs_n(image):
    'VIIRS band selection and conversion'
    lst = image.select('LST_1KM').subtract(273.15).rename('VIIRS_LST_N')
    return image.addBands(lst)



# Load MODIS Terra and Aqua data, apply quality control and conversion functions

ERA5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .select(['temperature_2m'])
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(era5_t2m)
)

MOD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    #.filterDate(date_start, date_end_mod)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_mod_day)
)

MOD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MOD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end)
    #.filterDate(date_start, date_end_mod)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_mod_night)
)

MYD11A1Daytime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Day_1km', 'QC_Day'])
    .filterDate(date_start, date_end)
    #.filterDate(date_start, date_end_myd)
    .filterBounds(greenland)
    .map(maskQualityDaytime)
    .map(lst_myd_day)
)

MYD11A1Nighttime = (
    ee.ImageCollection('MODIS/061/MYD11A1')
    .select(['LST_Night_1km', 'QC_Night'])
    .filterDate(date_start, date_end)
    #.filterDate(date_start, date_end_myd)
    .filterBounds(greenland)
    .map(maskQualityNighttime)
    .map(lst_myd_night)
)

JAXA_A = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'A')) # Filter for ascending (AM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) 
    .filterDate(date_start, date_end)
    .sort('system:time_start', True) 
    .filterBounds(greenland)
    .map(maskJaxa)
    .map(lst_jaxa_a)
)

JAXA_D = (
    ee.ImageCollection('JAXA/GCOM-C/L3/LAND/LST/V3')
    .select(['LST_AVE', 'LST_QA_flag'])
    .filter(ee.Filter.eq('SATELLITE_DIRECTION', 'D')) # Filter for descending (PM) overpasses
    .filter(ee.Filter.eq('PROCESSING_RESULT', 'Good')) 
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskJaxa)
    .map(lst_jaxa_d)
)


VIIRS_Day = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1D")
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskViirs)
    .map(lst_viirs_d)
)


VIIRS_Night = (
    ee.ImageCollection("NASA/VIIRS/002/VNP21A1N") 
    .filterDate(date_start, date_end)
    .filterBounds(greenland)
    .map(maskViirs)
    .map(lst_viirs_n)
)


In [11]:
# VIIRS_Day.first()   
#JAXA_A.first()
JAXA_D.first()